In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve, auc)


In [15]:
df = pd.read_csv(r'../merge/raw/csv_full_post.csv', sep=';')

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

pd.DataFrame({
    'null_count': null_counts,
    'null_%': null_pct
}).sort_values('null_%', ascending=False)

,null_count,null_%
taux_plein_teom,871,5.05
zone_emploi,135,0.78
prix_m2,19,0.11
annee,0,0.00
dep_nom,0,0.00
reg_code,0,0.00
code_postal,0,0.00
dep_code,0,0.00
population,0,0.00
reg_nom,0,0.00


In [16]:
df_model = df.drop(columns=['dep_nom', 'reg_nom'])

df_model.head()

,annee,code_postal,prix_m2,dep_code,reg_code,population,superficie_km2,zone_emploi,taux_global_tfb,taux_global_tfnb,taux_plein_teom,taux_global_th,nb_ventes,y
0,2022,1000.0,4903.185023,01,84.0,47054.0,37.0,8405.0,38.82,118.47,9.95,28.63,3232,stable
1,2022,1090.0,3208.893962,01,84.0,9048.0,41.0,8434.0,38.82,118.47,9.95,28.63,539,stable
2,2022,1100.0,5970.935347,01,84.0,33577.0,105.0,8425.0,38.82,118.47,9.95,28.63,1790,stable
3,2022,1110.0,5222.848885,01,84.0,6469.0,128.0,8404.0,38.82,118.47,9.95,28.63,596,stable
4,2022,1120.0,5424.557137,01,84.0,19078.0,93.0,8421.0,38.82,118.47,9.95,28.63,1119,stable


In [17]:
df_model.columns.to_list()

['annee',
 'code_postal',
 'prix_m2',
 'dep_code',
 'reg_code',
 'population',
 'superficie_km2',
 'zone_emploi',
 'taux_global_tfb',
 'taux_global_tfnb',
 'taux_plein_teom',
 'taux_global_th',
 'nb_ventes',
 'y']

In [18]:
df_model = df_model[~df_model['dep_code'].isin(['2A', '2B'])]

In [19]:
features = ['annee', 'dep_code', 'reg_code', 'code_postal', 'population', 'superficie_km2',
 'zone_emploi', 'taux_global_tfb', 'taux_global_tfnb', 'taux_plein_teom', 'taux_global_th','nb_ventes']

X = df_model[features]
y = df_model['y']

train = df_model[df_model['annee'] < 2024]
test  = df_model[df_model['annee'] == 2024]

X_train = train[features]
y_train = train['y']
X_test  = test[features]
y_test  = test['y']

print(f"Train : {len(X_train)} lignes")
print(f"Test  : {len(X_test)} lignes")
print(f"\nDistribution train :\n{y_train.value_counts()}")
print(f"\nDistribution test :\n{y_test.value_counts()}")

Train : 11278 lignes
Test  : 5640 lignes

Distribution train :
y
stable    5645
baisse    2987
hausse    2646
Name: count, dtype: int64

Distribution test :
y
baisse    3173
hausse    2460
stable       7
Name: count, dtype: int64


In [20]:
pipeline_simple = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

pipeline_simple.fit(X_train, y_train)

# Prédictions
y_pred = pipeline_simple.predict(X_test)

# Évaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"✓ Pipeline entraîné !")
print(f"Accuracy : {accuracy:.2%}")

✓ Pipeline entraîné !
Accuracy : 49.31%


In [21]:
print(classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred, labels=['hausse', 'baisse', 'stable'])


              precision    recall  f1-score   support

      baisse       0.55      0.54      0.54      3173
      hausse       0.42      0.43      0.43      2460
      stable       0.56      0.71      0.62         7

    accuracy                           0.49      5640
   macro avg       0.51      0.56      0.53      5640
weighted avg       0.49      0.49      0.49      5640

